# Aggregating Attribute Data


In the previous section, we aggregated point data by polygon — counting how many features fall within each spatial unit. But a count is often not enough: the **attributes** of those features matter too, and they can be summarised.

In this section, we will aggregate attribute data from a layer of apartment buildings in Saint Petersburg by municipal unit.

- We will perform a spatial join to assign each building to its municipal unit.
- We will group the data by unit and calculate statistics on the year of construction (mean, median, minimum, and maximum).
- We will join the results back to the polygon layer and visualise the output.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt  # plotting
import matplotlib as mpl         # core matplotlib module

### 0.2. Preparing the Data


This section uses two files from `data/spb/`:

- **spb_admin.gpkg** — boundaries of the city's districts and municipal units;
- **spb_mkd.csv** — apartment buildings in Saint Petersburg, 2020.

_Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._


We read the municipal unit boundaries of Saint Petersburg from the GeoPackage file.


In [ ]:
okrug = gpd.read_file("../../data/spb/spb_admin.gpkg", layer="okrug")

okrug.explore(tiles="cartodbpositron")

Then the apartment building data.


In [ ]:
mkd_csv = pd.read_csv("../../data/spb/spb_mkd.csv")

# Preview the data to understand which fields contain the coordinates
mkd_csv.head()

Coordinates are stored in a single `coordinates` field. We need to split it into two columns — `lat` (latitude) and `lon` (longitude) — and then create a GeoDataFrame from them.


In [ ]:
# Split the coordinates field into two columns.
# In this file the values are stored as "latitude,longitude" — always check the order,
# swapping them silently puts the data somewhere else on the map.
mkd_csv[["lat", "lon"]] = mkd_csv["coordinates"].str.split(",", expand=True).astype(float)

# Create a GeoDataFrame from the DataFrame using the coordinate columns.
# points_from_xy takes x (longitude) first, then y (latitude)
mkd_gdf = gpd.GeoDataFrame(
    mkd_csv,
    geometry=gpd.points_from_xy(mkd_csv["lon"], mkd_csv["lat"]),
    crs="EPSG:4326"
)

The first 100 features on a map. Rendering the full dataset is not recommended — the large number of features may prevent the map from loading.


In [ ]:
mkd_gdf.head(100).explore(tiles="cartodbpositron")

And the summary information for the GeoDataFrame:


In [ ]:
mkd_gdf.info()

We do not have documentation describing each field, but we can make reasonable inferences from the column names.

For this exercise, we will focus on the year of construction field — `data_buildingdate`. For each municipal unit, we will calculate the mean, median, earliest, and latest year of construction across all apartment buildings within it.


## 1. Spatial Join

First, we need to join the apartment buildings layer with the municipal unit boundaries. We perform a spatial join to determine which unit each building belongs to.


### 1.1. Checking the CRS


Do the two layers share a CRS?


In [ ]:
okrug.crs == mkd_gdf.crs

They match, so we can proceed without reprojecting.


### 1.2. Performing the Spatial Join


For each apartment building, we identify the municipal unit it falls within using `sjoin` with the `within` predicate. We use a left join (`how="left"`) to retain all buildings, including any that fall outside the unit boundaries.


In [ ]:
mkd_in_okrug = gpd.sjoin(
    mkd_gdf,
    okrug,
    how="left",
    predicate="within"
)

The spatial join appends the attributes of the municipal unit to each building.

The first five rows of the result:


In [ ]:
mkd_in_okrug.head()

## 2. Aggregating Attribute Data

Now that each building has been assigned to a municipal unit, we can analyse the attribute data. We will group buildings by unit and compute summary statistics for the year of construction field.


### 2.1. Checking the Data Type

Since we want to calculate numeric statistics, we need to verify that the `data_buildingdate` field is stored as a numeric type — and convert it if not.


In [ ]:
print(f"Data type: {mkd_in_okrug['data_buildingdate'].dtype}")

The field comes back as a string type (`str`; in pandas versions before 3.0 the same thing is shown as `object`), meaning the values are stored as text. We cannot compute numeric statistics in this format, so we need to convert the field to a numeric type.

We use `pd.to_numeric` with `errors='coerce'`, which converts non-numeric values to `NaN` rather than raising an error.

After the conversion, we verify the data type — the field should now be numeric.

In [ ]:
# Convert to numeric
mkd_in_okrug["data_buildingdate"] = pd.to_numeric(mkd_in_okrug["data_buildingdate"], errors="coerce")

# Verify the type after conversion
print(f"Data type: {mkd_in_okrug['data_buildingdate'].dtype}")

The field is now numeric, so we can compute the statistics.


### 2.2. Computing Summary Statistics


Let's calculate the mean year of construction per municipal unit and join the result back to the polygon layer.

We write the result into a new layer, `okrug_stats`, and leave `okrug` untouched — that way the cells below can be re-run in any order without joining the same columns twice.

In [ ]:
# Group by unit and calculate mean year of construction
build_year_mean = mkd_in_okrug.groupby("NAME")["data_buildingdate"].mean()

# Join the result to the municipal units layer
okrug_stats = okrug.merge(
    build_year_mean.rename("build_year_mean"),
    on="NAME",
    how="left"
)

The result on the map:

In [ ]:
okrug_stats.explore(column="build_year_mean", cmap="YlGnBu", tiles="cartodbpositron")

Now let's compute the median, earliest, and latest year of construction in the same way.


In [ ]:
# Calculate median, minimum, and maximum year of construction per unit
build_year_median = mkd_in_okrug.groupby("NAME")["data_buildingdate"].median()
build_year_min = mkd_in_okrug.groupby("NAME")["data_buildingdate"].min()
build_year_max = mkd_in_okrug.groupby("NAME")["data_buildingdate"].max()

# Combine all statistics into a single table
build_year_stats = pd.DataFrame({
    "build_year_median": build_year_median,
    "build_year_min": build_year_min,
    "build_year_max": build_year_max
}).reset_index()

# Join the results to the municipal units layer on NAME
okrug_stats = okrug_stats.merge(build_year_stats, on="NAME", how="left")

_Here we group on the NAME field. In practice, group on a unique identifier instead — names repeat and get spelled differently._


Check that the new columns are in place:


In [ ]:
# Preview the first few rows
okrug_stats[["build_year_mean", "build_year_median", "build_year_max", "build_year_min"]].head()

Now all four statistics on separate maps, side by side.


## 3. Choropleth Maps

Let's produce four maps and compare the statistics side by side. To make the comparison meaningful, we will use a shared colour scale across all maps.


### 3.1. Defining the Metrics


We define the list of metrics to display. For each one, we specify:

- the field name in the data
- the map title


In [ ]:
metrics = [
    ("build_year_mean",   "Mean year of construction"),
    ("build_year_median", "Median year of construction"),
    ("build_year_min",    "Earliest year of construction"),
    ("build_year_max",    "Latest year of construction"),
]

### 3.2. Setting a Shared Colour Scale


We find the minimum and maximum values across all metrics.

This ensures that:

- all maps use the **same colour range**
- values can be meaningfully compared across maps


In [ ]:
# Shared value range for all maps
vmin = okrug_stats[[m for m, _ in metrics]].min().min()
vmax = okrug_stats[[m for m, _ in metrics]].max().max()

### 3.3. Creating the Maps

One map per metric, in a single row, on the shared colour scale.


In [ ]:
# Create a figure with 4 subplots in a single row
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)

# Set up the colour scale
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)  # normalise values
cmap = mpl.colormaps["viridis"]  # colour palette

# Plot a map for each metric
for ax, (metric, title) in zip(axes, metrics):
    okrug_stats.plot(
        column=metric,      # metric to display
        ax=ax,              # target subplot
        cmap=cmap,          # colour palette
        norm=norm,          # shared value range
        linewidth=0.5,
        edgecolor="gray"
    )
    ax.set_title(title, fontsize=12)  # map title
    ax.axis("off")  # hide axes

# Add a shared colour bar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)

cbar = fig.colorbar(
    sm,
    ax=axes.tolist(),
    orientation="horizontal",
    fraction=0.05,
    pad=0.02
)
cbar.set_label("Year of construction", fontsize=12)

# Display the result
plt.show()

## Summary


In this section, we learned how to aggregate **attribute data** from a spatial dataset and analyse it by spatial unit.

We covered:

- how to use a spatial join to assign each feature to a spatial unit;
- how to group data by spatial unit and compute summary statistics.

In this example, we analysed the year of construction of apartment buildings. The same approach applies to any numeric attribute data.
